## ❌ Error Troubleshooting: JreNotFound (Java Runtime Environment)

### Why did this error happen?
You encountered the `JreNotFound` error because you selected **Parquet** as your destination (sink) format. When using a **Self-Hosted Integration Runtime** to read or write Parquet (or ORC) files, Azure Data Factory requires a Java Runtime Environment (JRE) to be installed on the local machine where the Integration Runtime is running. Parquet serialization relies on Java under the hood.

### 🛠️ How to Fix the "JreNotFound" Error

Follow these steps on the local machine/VM where your Self-Hosted Integration Runtime is installed:

**Step 1: Download and Install Java**
You need to install either Oracle JRE 8 or a compatible OpenJDK (like Eclipse Temurin or Zulu OpenJDK). 
1. Download a 64-bit Java Runtime Environment (e.g., [Eclipse Temurin OpenJDK 11](https://adoptium.net/) or Oracle JRE 8).
2. Run the installer and complete the installation on your local system.

**Step 2: Set the JAVA_HOME Environment Variable**
Your system needs to know exactly where Java is installed.
1. Open the Windows Start Menu and search for **"Environment Variables"** (Select *Edit the system environment variables*).
2. Click on the **Environment Variables...** button at the bottom.
3. Under the **System variables** section (the bottom half), click **New...**
4. Set the **Variable name** to: `JAVA_HOME`
5. Set the **Variable value** to your Java installation path (e.g., `C:\Program Files\Eclipse Adoptium\jre-11.0.18.10-hotspot` or `C:\Program Files\Java\jre1.8.0_321`). Click OK.

**Step 3: Update the Path Variable**
1. Still in the **System variables** section, find the variable named **Path** and click **Edit...**
2. Click **New** and add the following: `%JAVA_HOME%\bin`
3. Click **OK** on all windows to save the changes.

**Step 4: Restart the Integration Runtime Service**
For the Self-Hosted Integration Runtime to recognize the new Java installation, it must be restarted.
1. Open the Windows Start Menu and search for **Services** (or `services.msc`).
2. Scroll down and find **Azure Data Factory Integration Runtime Service**.
3. Right-click it and select **Restart**.

**Step 5: Retry the Pipeline**
* Go back to Azure Data Factory.
* Click **Debug** on your `onprem_ingestion` pipeline again. 
* ✅ **Result**: The pipeline should now successfully serialize your CSV data into Parquet format and land it in your Data Lake!

## ❌ Error Troubleshooting: MappingColumnNameNotFoundInSourceFile

### Why did this error happen?
You encountered the `MappingColumnNameNotFoundInSourceFile` error because your **Copy Data** activity is still holding onto a hardcoded schema mapping from a previous test, but you are now passing *different* files dynamically through a `ForEach` loop. 

When ADF tries to copy the next file in your list (e.g., `DimAirline.csv`), it looks for the exact column names that were saved in the Mapping tab from your first file. Since the new file has different columns, ADF throws this error because it cannot find the expected column names.

### 🛠️ How to Fix It (Enable Dynamic Mapping)

To fix this, we need to tell the Copy Activity to ignore any hardcoded mappings and instead dynamically read the schema of whatever file is currently being processed.

**Step 1: Locate the Copy Activity**
1. Open your `onprem_ingestion` pipeline.
2. Double-click the **ForEach** activity to open its inner canvas.
3. Click on the **Copy Data** (`MigrateOnPremFile`) activity inside.

**Step 2: Clear the Explicit Mapping**
1. Look at the properties pane at the bottom and click on the **Mapping** tab.
2. If you see a list of columns mapped out here, click the **Clear mapping** button (often a trash can icon or a "Clear" button at the top of that section). 
3. The mapping area should now be completely empty. When left empty, ADF automatically maps all source columns to the sink based on the headers of the current file.

**Step 3: Verify Source Dataset Headers (Crucial for CSVs)**
For auto-mapping to work correctly with CSV files, ADF needs to know that the first row contains the column names.
1. Go to the **Source** tab of your Copy Activity and open your source dataset (`ds_onpre_src_param`).
2. Make sure the checkbox for **First row as header** is **checked**. (Do the same for your Sink dataset if you want the CSV headers to appear in the destination).

**Step 4: Retry the Pipeline**
1. Go back to the main pipeline canvas.
2. Click **Validate** to ensure everything is saved.
3. Click **Debug** to run the pipeline again.
4. ✅ **Result**: The pipeline will now dynamically read the headers of `Dimpassenger.csv`, `DimAirline.csv`, and `DimFlight.csv` on the fly, and the error will be resolved!